# Descarga de imágenes - Ejemplo práctico

Objetivo:
- Leer el catálogo oficial de cámaras.
- Descargar una imagen de cada cámara desde `URLIMAGEN`.
- Almacenar las capturas en formato reproducible (`raw/<camara>/<YYYYMMDDTHHMMSSZ>.jpg`, hora UTC).
- Generar un fichero de metadatos.

La captura continua de las cámaras seleccionadas la realiza el servicio `scripts/captura_camaras.py`; este notebook sirve para obtener una instantánea de **todas** las cámaras del catálogo.

In [ ]:
import logging
import sys
from datetime import datetime, timezone
from pathlib import Path
from time import sleep

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from catalogo import load_catalog
from config import load_config, path
from tiempo import FILENAME_FORMAT

CFG = load_config()
RAW_DIR = path("raw")
METADATA_DIR = path("metadata")
RAW_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

REQUEST_TIMEOUT = CFG["captura"]["timeout_s"]
SLEEP_SECONDS = 1

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger()

In [ ]:
# El catálogo ya limpia URLIMAGEN (algunas traen un tabulador final que provocaba un 404)
df = load_catalog()
print(f"Cámaras encontradas: {len(df)}")

required_columns = ["ID", "NOMBRE", "URLIMAGEN"]
for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Falta columna {col}")
print("Validación correcta")

df.head()

In [ ]:
# Sesión global con reintentos y espera exponencial (1 s, 2 s, 4 s...)
session = requests.Session()
retries = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retries))

# Cabeceras indispensables para evitar el error 403 Forbidden
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://malaga.eu",
    "Accept": "image/avif,image/webp,image/apng,image/svg+xml,image/*,*/*;q=0.8",
    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
}


def download_image(image_url, output_file):
    response = session.get(image_url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    output_file.write_bytes(response.content)


def capture_camera(row):
    camera_id = row["ID"]
    camera_name = str(row["NOMBRE"])
    image_url = row["URLIMAGEN"]

    if pd.isna(image_url):
        return None

    # Nombre de fichero en UTC, igual que captura_camaras.py
    timestamp = datetime.now(timezone.utc).strftime(FILENAME_FORMAT)
    camera_dir = RAW_DIR / camera_id
    camera_dir.mkdir(exist_ok=True)
    image_file = camera_dir / f"{timestamp}.jpg"

    try:
        download_image(image_url, image_file)
        status, saved = "OK", image_file.relative_to(path("datos")).as_posix()
    except Exception as e:
        logger.error(f"{camera_name}: {e}")
        status, saved = f"ERROR: {e}", None

    return {
        "camera_id": camera_id,
        "camera_name": camera_name,
        "image_url": image_url,
        "capture_time": timestamp,
        "image_file": saved,
        "status": status,
    }

In [ ]:
results = []
for _, row in df.iterrows():
    result = capture_camera(row)
    if result:
        results.append(result)
    sleep(SLEEP_SECONDS)

captures_df = pd.DataFrame(results)
captures_df["status"].str.startswith("OK").value_counts()

In [ ]:
metadata_file = METADATA_DIR / "captura_imagenes.csv"
captures_df.to_csv(metadata_file, index=False)
print(f"Metadatos guardados en: {metadata_file}")

captures_df.sample(min(10, len(captures_df)))